In [3]:
import os, sys
from pathlib import Path
import numpy as np

# os.environ["CUDA_VISIBLE_DEVICES"] = ""

try:
    __file__
except NameError:
    __file__ = str(Path(os.getcwd())/'notebook.ipynb')  # Convert Path to string
base_dir = os.path.dirname(os.path.abspath(__file__))
# The base_dir variable points to the current directory
parent_dir = os.path.dirname(base_dir)  # Get the parent directory
sys.path.append(parent_dir)  # Add the parent directory to the Python path

from utils.plot_function import AcademicPlot
# Instantiate the plotting class
plotter = AcademicPlot(style='seaborn-v0_8-paper', figsize=(2, 1.5))

%load_ext autoreload
%autoreload 2

# import torch
# torch.cuda.is_available = lambda: False  # Force PyTorch to think CUDA is not available
# device = torch.('cpu')  # Explicitly set device to CPU


In [ ]:
"""load data"""

from data.data_process.data_loader_multi_wecc import prepare_data

fault_idx = [1,2,3,4,5,6,7,8,9,11,14] # fault_idx = [1,2,3,4,5,6,7,8,9,10,11,14]
gen_buses = [1,2,3,6,8]
data_folder_path = os.path.abspath(os.path.join(base_dir, ".."))+os.sep+'data/data_folder/IEEE14_1'
N_traj = len(fault_idx)

# load the data set
# data12 = prepare_data(data_folder_path, fault_idx, 'ieee14_fault_1.2', 1.2)
data = prepare_data(data_folder_path, fault_idx, 'ieee14_fault_1.1', 1.1, phasor_buses = gen_buses)

# seperate timeseries
omega_list =  data['omega_list']
delta_list =  data['delta_list']
time_list =  data['time_list']

# Dataset splitting
np.random.seed(0)  # Set random seed for reproducibility
all_indices = np.arange(N_traj)
np.random.shuffle(all_indices)

# Split dataset
n_train = 9  # Training set size

# Get indices for training and test sets
train_indices = all_indices[:n_train ].tolist()  # First part for training
test_indices = all_indices[n_train:].tolist()    # Remaining for testing

train_Ntraj = len(train_indices)

# Create input data lists with conversion
# Reshape data to [n_features, n_timesteps] * n_trajectories
train_x = [np.concatenate((
    (omega_list[ii]-1)*2*np.pi*60,
    delta_list[ii],
    # data['pe_list'][ii],
    # data['voltage_list'][ii]
), axis=1) for ii in train_indices]          
n_vars_x = train_x[0].shape[-1]                   # = omega + delta + pe

# Reshape data to [n_features, n_timesteps] * n_trajectories
test_x = [np.concatenate((
    (omega_list[ii]-1)*2*np.pi*60,
    delta_list[ii],
    # data['pe_list'][ii],
    # data['voltage_list'][ii]
), axis=1) for ii in test_indices]          # Shape: [n_features, n_timesteps] * n_trajectories

# Print dataset splitting info
print(f"Total trajectories: {N_traj}")
print(f"Training trajectories per fold: {len(train_indices)}")
# print(f"Test trajectories: {len(test_indices)}")

# Similarly process time data
train_time = [time_list[ii] for ii in train_indices]
test_time = [time_list[ii] for ii in test_indices]

# Print data shapes for verification
print("Training data shape:", train_x[0].shape)  # [n_features, n_timesteps] * n_train_trajectories
print("Test data shape:", test_x[0].shape)       # [n_features, n_timesteps] * n_test_trajectories

print("Number of training trajectories:", len(train_x))
print("Number of test trajectories:", len(test_x))


In [ ]:
import sys
sys.path.append('../src')

import deepkoopman
import numpy as np
# import numpy.random as rnd
np.random.seed(42)  # for reproducibility

# Explicitly disable CUDA in PyTorch
import torch
# torch.cuda.is_available = lambda: False  # Force PyTorch to think CUDA is not available
# device = torch.device('cpu')  # Explicitly set device to CPU
# torch.set_float32_matmul_precision('high')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#torch.save into pth 
from deepkoopman.regression import DeepKoopman

data_dt = data['time_list'][0][1] - data['time_list'][0][0]
state_dim = 10
time_delay_embed = 3
extension_dim = 16
inn_type = "freiaallinone"

# Create the two-stage model
dlk_regressor = DeepKoopman(
    dt=data_dt,
    look_forward=8,
    time_delay=time_delay_embed,
    config_inn=dict(
        input_size=state_dim * time_delay_embed,
        hidden_size=20 * time_delay_embed,
        num_layers=2,
        output_size=state_dim * time_delay_embed,
        init_identity=True,
        inn_type = inn_type,
    ),
    extension_config=dict(
        extension_type = "cnn",
        extension_output_size = 16,
        hidden_channels=[32, 64],
        kernel_sizes=[3, 5],
        dropout_rate=0.1,
        use_residual=True
    ), 
    batch_size=256,
    normalize=True,
    # extension_hidden_size=32,       # Hidden size of the extension layer
    # extension_num_layers=3,         # Number of layers in the extension layer
    progressive_steps=True,         # Whether to use progressive prediction steps
    trainer_kwargs={'max_epochs': 5, 
                    'accelerator': 'cpu'},
)

# Create directory to save training results
os.makedirs('deephodmd_results', exist_ok=True)

# Train the model
dlk_regressor.fit(
    x=train_x,
    y=None,
    monitor_params=True,
    log_dir='deephodmd_results',
    record_losses=True
)


# Nach dem Training: Modell & Konfig sichern
"""
save_dict = {
    "inn_state_dict": dlk_regressor._regressor.state_dict(),
    "inn_type": inn_type,
    "config_inn": dlk_regressor.config_inn,
    "look_forward": dlk_regressor.look_forward,
    "time_delay": dlk_regressor.time_delay,
    "losses": getattr(dlk_regressor, "losses_", None),
    "eigenvalues": dlk_regressor._eigenvalues_,
    "dm": dlk_regressor.dm
}

save_path = f"deephodmd_results/dlk_{inn_type}2.pth"
torch.save(save_dict, save_path)
print(f"✅ Modell erfolgreich gespeichert unter: {save_path}")
"""



In [ ]:
rmse_error = [None]*train_Ntraj
rrmse_factor = [None]*train_Ntraj
for ii in range(train_Ntraj):
    x0 = train_x[ii][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
    t = train_time[ii]-train_time[ii][0]
    sol = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)[:,:state_dim]  

    rmse_error[ii]  = np.linalg.norm(train_x[ii][:,:state_dim] - sol.real)**2 
    rrmse_factor[ii] = np.linalg.norm(train_x[ii][:,:state_dim])**2

print("Absolute error: ", np.sqrt(sum(rmse_error)))
print("Absolute error: ", np.sqrt(sum(rmse_error)/sum(rrmse_factor))*100, "%")

In [ ]:
import matplotlib.pyplot as plt

ii = 0
x0 = train_x[ii][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
t = train_time[ii]-train_time[ii][0]
sol = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)[:,:state_dim] # shape:(sequences, feature)

rmse_error[ii]  = np.linalg.norm(train_x[ii][:,:state_dim] - sol.real)**2 
rrmse_factor[ii] = np.linalg.norm(train_x[ii][:,:state_dim])**2

Test on training data

In [ ]:
train_idx = 5

x0 = train_x[train_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)

X_train = train_x[train_idx][:len(t)]
# X_train = np.vstack([x0[np.newaxis, :], X_train])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
train_traj_idx = train_indices[train_idx]
time_train_complete =  data['origin_time'][train_traj_idx]
omega_train_complete = data['origin_omega'][train_traj_idx]

# Plotting
for var_id in [0,1,2,3]:
    plotter.plot(
        [time_train_complete,            t+train_time[train_idx][0]],
        [omega_train_complete[:,var_id], sol[:,var_id].real   ], 
        # scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10),                
        xticks=np.linspace(0, 10, 6),
        plot_legend = True,
        plot_grid= False
    )

In [ ]:
test_idx = 0
x0 = test_x[test_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)
Xtest = test_x[test_idx][:len(t)]
# Xtest = np.vstack([x0[np.newaxis, :], Xtest])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
test_traj_idx = test_indices[test_idx]
time_test_complete =  data['origin_time'][test_traj_idx]
omega_test_complete = data['origin_omega'][test_traj_idx]

# plot the curve
for var_id in [0,1,2,3]:
    plotter.plot(
        [time_test_complete,            t+test_time[test_idx][0]],
        [omega_test_complete[:,var_id], sol[:,var_id].real   ], 
        # scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10), 
        xticks=np.linspace(0, 10, 6),
        plot_legend = True,
        plot_grid= False
    )

In [ ]:
test_idx = 1
x0 = test_x[test_idx][0:time_delay_embed].reshape(-1)  #np.array([2, -4])
T = 10
t = np.arange(0, T, data_dt)
Xtest = test_x[test_idx][:len(t)]
# Xtest = np.vstack([x0[np.newaxis, :], Xtest])

Xkoop_nn = dlk_regressor.simulate(x0[np.newaxis, :], n_steps=len(t)-1)

sol = Xkoop_nn/2/np.pi/60 + 1
test_traj_idx = test_indices[test_idx]
time_test_complete =  data['origin_time'][test_traj_idx]
omega_test_complete = data['origin_omega'][test_traj_idx]


# plot the curve
for var_id in [0,1,2,3]:
    plotter.plot(
        [time_test_complete,            t+test_time[test_idx][0]],
        [omega_test_complete[:,var_id], sol[:,var_id].real   ], 
        # scatter_data = scatter_points,
        labels=[r"Truth", r"Deep DMD"], 
        xlabel="Time (s)", 
        ylabel="Omega of Gen \n on Bus " + str(gen_buses[var_id]) + " (p.u.)", 
        title=None, 
        linestyles = ['-','-'],
        filename=None,
        xlim=(0, 10),       
        xticks=np.linspace(0, 10, 6),
        plot_legend = True,
        plot_grid= False
    )

Testing on unseen trajectory

In [ ]:
dlk_regressor._eigenvalues_

In [ ]:
psi_0 = dlk_regressor._compute_psi(x0)  # (n_koopman, n_samples)
T = 10      
t =  np.arange(0, T, data_dt)
# Advance using eigenvalues
psi_t = np.zeros((state_dim*time_delay_embed + extension_dim, len(t)))                  # (n_koopman, n_samples)
for tt in range(len(t)):
    psi_t[:, tt] = (np.diag(np.power(dlk_regressor.eigenvalues_, tt)) @ psi_0)[:,-1]  # (n_koopman, n_samples)

In [ ]:
# 绘制曲线
for var_id in [3]:
    plotter.plot(
        [t],
        [psi_t[var_id]], 
        labels=[r"Deep DMD"], 
        xlabel="Time (s)", 
        title=None, 
        linestyles = ['-'],
        filename=None,
        xlim=(0, T), 
        xticks=np.linspace(0, 10, 6),
        plot_legend = True,
        plot_grid= False
    )

In [ ]:
abs(dlk_regressor.eigenvalues_)

In [ ]:
from plot_function import plot_eigs
plot_eigs(dlk_regressor.eigenvalues_)

In [ ]:
eigv = np.log(dlk_regressor.eigenvalues_)/data_dt
print(eigv)

In [ ]:
mode_i =0
tmp = dlk_regressor._eigenvectors_[:,[mode_i]] @ dlk_regressor.psi(train_x[train_idx][0:time_delay_embed].reshape(-1))[[mode_i],:]
print(tmp)
print(np.linalg.norm(tmp))